In [1]:
import os
import json
import random
import time
import requests
import pandas as pd

API_URL = "https://ckey.vn/v1/chat/completions"
API_KEY = "sk-824a080f44c7c8de2bd15ff4e13d386b4581ecc54ae1d506b049183d13a8c808"
MODELS = ["gpt-5.4-mini", "gpt-5.4", "gpt-5.5"]
FILE_NAME = "train_5_classes.csv"
BATCH_SIZE = 50
TOTAL_BATCHES = 80
MAX_RETRIES = 3

SYSTEM_PROMPT = """You are a strictly constrained data synthesizer for a fashion RAG intent classifier.
Generate diverse user queries. Output ONLY a valid JSON array of objects: [{"query": "...", "intent": "..."}].

DISTRIBUTION REQUIREMENTS (STRICT 50 QUERIES PER BATCH, 10 per class):
Classify EVERY query into exactly ONE of these 5 strict labels: 
'similar_items', 'graph_pairing', 'color_variant', 'chit_chat', 'composite_intent'.

1. 'similar_items' (10 queries)
- Pure noun phrases, dupe requests, or looking for replacements.
- MUST NOT contain verbs/context about matching or occasions.
- e.g., "wide leg jeans", "dupe for this", "cheaper alternative".

2. 'graph_pairing' (10 queries)
- Asking for COMPLEMENTARY items to wear WITH the anchor item.
- e.g., "pants to match this", "accessories for this look", "what to wear for a beach party".

3. 'color_variant' (10 queries)
- Change color ONLY.
- e.g., "do you have it in black", "red version pls".

4. 'chit_chat' (10 queries)
- Greetings, weather, order status, or nonsense.
- e.g., "hello", "where is my order", "skrrt banana".

5. 'composite_intent' (10 queries) - THE AMBIGUOUS CLASS
- Queries that contain MULTIPLE overlapping desires in a single sentence, forcing the system to ask for clarification.
- Mix 'similar_items' + 'graph_pairing': "Find a cheaper dupe for this jacket to wear with my boots."
- Mix 'similar_items' + 'color_variant': "Show me lookalikes of this bag but in red."
- Mix 'color_variant' + 'graph_pairing': "Do you have this exact shirt in navy to match these pants?"
- Highly complex requests: "Cheaper version of this in black, and show me shoes to wear with it."
"""

FASHION_TOPICS = [
    "winter coats", "swimwear", "formal wedding", "streetwear", "vintage 90s",
    "activewear", "office suits", "summer dresses", "luxury handbags", "goth boots",
    "minimalist wardrobe", "boho chic", "athleisure", "preppy knitwear",
    "cyberpunk techwear", "old money aesthetic", "loungewear", "festival outfits",
    "rainy day gear", "maternity fashion", "chunky loafers", "corsets"
]

print('setup loaded')

setup loaded


In [2]:
print('generation started')

for i in range(1, TOTAL_BATCHES + 1):
    topic = random.choice(FASHION_TOPICS)
    current_temp = round(random.uniform(0.75, 0.95), 2)
    current_model = random.choice(MODELS)
    seed_word = random.choice(["lazy typing", "slang", "polite", "urgent", "hyper-specific", "vague", "typos included"])
    
    prompt_modifier = f"Batch ID: {i}. Topic: '{topic}'. User persona/typing style: '{seed_word}'. Ensure vocabulary is completely different from previous batches."
    
    payload = {
        "model": current_model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_modifier}
        ],
        "temperature": current_temp,
        "max_tokens": 2500
    }
    
    success = False
    retries = 0
    
    while not success and retries < MAX_RETRIES:
        try:
            response = requests.post(
                API_URL, 
                headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}, 
                json=payload, 
                timeout=60
            )
            
            if response.status_code == 200:
                raw_text = response.json()['choices'][0]['message']['content']
                clean_json = raw_text.strip().replace("```json", "").replace("```", "").strip()
                
                data = json.loads(clean_json)
                df = pd.DataFrame(data)
                
                write_header = not os.path.exists(FILE_NAME)
                df.to_csv(FILE_NAME, mode='a', index=False, header=write_header, encoding='utf-8')
                
                print(f'batch {i} completed')
                success = True
            else:
                retries += 1
                time.sleep(2)
                
        except Exception:
            retries += 1
            time.sleep(2)

print('generation finished')

generation started
batch 1 completed
batch 2 completed
batch 3 completed
batch 4 completed
batch 5 completed
batch 6 completed
batch 7 completed
batch 8 completed
batch 9 completed
batch 10 completed
batch 11 completed
batch 12 completed
batch 13 completed
batch 14 completed
batch 15 completed
batch 16 completed
batch 17 completed
batch 18 completed
batch 19 completed
batch 20 completed
batch 21 completed
batch 22 completed
batch 23 completed
batch 24 completed
batch 25 completed
batch 26 completed
batch 27 completed
batch 28 completed
batch 29 completed
batch 30 completed
batch 31 completed
batch 32 completed
batch 33 completed
batch 34 completed
batch 35 completed
batch 36 completed
batch 37 completed
batch 38 completed
batch 39 completed
batch 40 completed
batch 41 completed
batch 42 completed
batch 43 completed
batch 44 completed
batch 45 completed
batch 46 completed
batch 47 completed
batch 48 completed
batch 49 completed
batch 50 completed
batch 51 completed
batch 52 completed
ba

In [3]:
print('deduplication started')

df = pd.read_csv(FILE_NAME)
initial_count = len(df)

df['query_clean'] = df['query'].str.lower().str.strip()
df = df.drop_duplicates(subset=['query_clean', 'intent'], keep='first')
df = df.drop(columns=['query_clean'])

df.to_csv(FILE_NAME, index=False)
final_count = len(df)

print(f'initial rows: {initial_count}')
print(f'final rows: {final_count}')
print(df['intent'].value_counts())

deduplication started
initial rows: 4000
final rows: 3630
intent
composite_intent    800
graph_pairing       796
similar_items       778
color_variant       759
chit_chat           497
Name: count, dtype: int64


In [4]:
import pandas as pd
import requests
import json
import time
import os

print('evaluation started')

EVAL_MODEL = "gpt-5.4-mini"
INPUT_FILE = "train_5_classes.csv"
OUTPUT_FILE = "train_5_classes_evaluated.csv"
BATCH_SIZE = 50

EVAL_PROMPT = """You are an absolute threshold evaluator for a multi-class intent router.
Classify each query into exactly ONE of the following 5 classes: 'similar_items', 'graph_pairing', 'color_variant', 'chit_chat', 'composite_intent'.
Output ONLY a valid JSON array: [{"index": integer, "predicted_intent": "string"}].

EVALUATION BOUNDARIES:
- Greeting, general question, weather, nonsense -> 'chit_chat'.
- Ask for COMPLEMENTARY items to wear WITH the anchor item (e.g., "pants to match this", "what to wear for a beach party") -> 'graph_pairing'.
- Direct clothing noun phrase (e.g., "red shirt") OR ask for a visual lookalike/dupe -> 'similar_items'.
- Ask for a different color of the same item -> 'color_variant'.
- IF the query contains MULTIPLE overlapping desires (e.g., asking for a similar/cheaper item AND something to wear with it, OR a different color AND shoes to match) -> 'composite_intent'.
"""

df_eval = pd.read_csv(INPUT_FILE)
df_eval['predicted_intent'] = None
df_eval['is_match'] = None

for i in range(0, len(df_eval), BATCH_SIZE):
    batch = df_eval.iloc[i:i+BATCH_SIZE]
    queries_payload = [{"index": int(idx), "query": row['query']} for idx, row in batch.iterrows()]
    
    payload = {
        "model": EVAL_MODEL,
        "messages": [
            {"role": "system", "content": EVAL_PROMPT},
            {"role": "user", "content": f"Evaluate:\n{json.dumps(queries_payload, ensure_ascii=False)}"}
        ],
        "temperature": 0.0, 
        "max_tokens": 2000
    }
    
    success = False
    retries = 0
    while not success and retries < MAX_RETRIES:
        try:
            response = requests.post(API_URL, headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}, json=payload, timeout=60)
            if response.status_code == 200:
                raw_text = response.json()['choices'][0]['message']['content']
                clean_json = raw_text.strip().replace("```json", "").replace("```", "").strip()
                results = json.loads(clean_json)
                
                for res in results:
                    idx = res['index']
                    pred = res['predicted_intent']
                    df_eval.at[idx, 'predicted_intent'] = pred
                    df_eval.at[idx, 'is_match'] = 1 if pred == df_eval.at[idx, 'intent'] else 0
                    
                success = True
                print(f"evaluated batch {i//BATCH_SIZE + 1}")
            else:
                retries += 1
                time.sleep(2)
        except Exception as e:
            retries += 1
            time.sleep(2)

df_eval.to_csv(OUTPUT_FILE, index=False)

matched = (df_eval['is_match'] == 1).sum()
total = len(df_eval)
print(f"\ntotal samples: {total}")
print(f"agreement rate: {matched}/{total} ({matched/total*100:.2f}%)")
print("mismatched samples:")
print(df_eval[df_eval['is_match'] == 0][['query', 'intent', 'predicted_intent']].head(10))

evaluation started
evaluated batch 1
evaluated batch 2
evaluated batch 3
evaluated batch 4
evaluated batch 5
evaluated batch 6
evaluated batch 7
evaluated batch 8
evaluated batch 9
evaluated batch 10
evaluated batch 11
evaluated batch 12
evaluated batch 13
evaluated batch 14
evaluated batch 15
evaluated batch 16
evaluated batch 17
evaluated batch 18
evaluated batch 19
evaluated batch 20
evaluated batch 21
evaluated batch 22
evaluated batch 23
evaluated batch 24
evaluated batch 25
evaluated batch 26
evaluated batch 27
evaluated batch 28
evaluated batch 29
evaluated batch 30
evaluated batch 31
evaluated batch 32
evaluated batch 33
evaluated batch 34
evaluated batch 35
evaluated batch 36
evaluated batch 37
evaluated batch 38
evaluated batch 39
evaluated batch 40
evaluated batch 41
evaluated batch 42
evaluated batch 43
evaluated batch 44
evaluated batch 45
evaluated batch 46
evaluated batch 47
evaluated batch 48
evaluated batch 49
evaluated batch 50
evaluated batch 51
evaluated batch 52
ev

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("train_5_classes_evaluated.csv")

df_clean = df[df['is_match'] == 1].copy()
df_clean = df_clean[['query', 'intent']]

train_df, temp_df = train_test_split(
    df_clean, 
    test_size=0.2, 
    stratify=df_clean['intent'], 
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    stratify=temp_df['intent'], 
    random_state=42
)

# 4. Lưu ra file
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)

print(f"total clean samples: {len(df_clean)}")
print(f"train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")

total clean samples: 3589
train: 2871 | val: 359 | test: 359
